# Multimodal Agentic Application — Part 1

Architecture and setup. Python 3.13; install `python -m pip install -r requirements.txt` from this series folder. Run from the companion repository. This notebook has no paid calls. [Part 2](./Part_2_Workflow.ipynb) assembles and runs the workflow. Never commit your `.env`.

In [ ]:
from pathlib import Path
import os,sys,tempfile,json
folder=Path.cwd()
if not (folder/'config.py').exists():
    folder=folder/'series'/'multimodal-agentic-workflow'
assert (folder/'config.py').is_file(), 'Open this notebook from the companion repository.'
sys.path.insert(0,str(folder))
from config import make_llm
print('Application files:', all((folder/f).is_file() for f in ['config.py','helper_tools.py','agentic_tools.py','agents.py','tasks.py','main.py']))

In [ ]:
from dotenv import dotenv_values
example=dotenv_values(folder/'.env.example')
assert example['RUN_OPENAI']=='0'
assert example['OPENAI_API_KEY']=='replace-with-your-own-key'
assert '.env' in (folder/'.gitignore').read_text().splitlines()
print('Explicit opt-in and ignored local credentials configured.')

In [ ]:
from agentic_tools import Route,RouterTool
from pydantic import ValidationError
validator=RouterTool()
for name in ['text2image','text2speech','image2text','web_search']:
    assert Route.model_validate_json(validator._run(json.dumps({'tool':name}))).tool==name
for value in ['{"tool":"delete_files"}','{"tool":"unsupported"}','{"tool":"web_search","extra":1}','not json']:
    try:validator._run(value)
    except (ValueError,ValidationError):pass
    else:raise AssertionError('Invalid route was accepted')
print('Four valid routes accepted; invalid/unsupported routes rejected without helper execution.')

In [ ]:
from PIL import Image
from helper_tools import image_bytes
with tempfile.TemporaryDirectory() as tmp:
    valid=Path(tmp)/'input.png';Image.new('RGB',(64,64),'blue').save(valid)
    data,mime=image_bytes(valid)
    assert mime=='image/png' and data==valid.read_bytes()
    invalid=Path(tmp)/'invalid.png';invalid.write_bytes(b'not an image')
    try:image_bytes(invalid)
    except (OSError,ValueError):pass
    else:raise AssertionError('Invalid image accepted')
print('Local image decoding and invalid-byte rejection passed.')

In [ ]:
before=os.environ.get('RUN_OPENAI')
os.environ['RUN_OPENAI']='0'
try:
    try:make_llm()
    except RuntimeError:print('Inference stays disabled without explicit opt-in.')
    else:raise AssertionError('Unexpected inference client')
finally:
    if before is None:os.environ.pop('RUN_OPENAI',None)
    else:os.environ['RUN_OPENAI']=before

## Original archive appendix

Original material below is preserved for comparison, not executed.

### Original block 1

````text
python -m venv crew-venv
````

### Original block 2

````text
.\crew-venv\Scripts\activate
````

### Original block 3

````text
source crew-venv/bin/activate
````

### Original block 4

````text
crewai>=0.28.0
langchain-groq>=0.1.0
langchain-community>=0.0.20
replicate>=0.22.0
python-dotenv>=1.0.0
tavily-python>=0.3.0
crewai-tools
````

### Original block 5

````text
pip install -r requirements.txt
````

### Original block 6

````text
OPENAI_API_KEY=your_openai_api_key_here
REPLICATE_API_TOKEN=your_replicate_api_token_here
TAVILY_API_KEY=your_tavily_api_key_here
GROQ_API_KEY=your_groq_api_key_here```
````

### Original block 7

````text
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
REPLICATE_API_TOKEN = os.environ.get('REPLICATE_API_TOKEN')
TAVILY_API_KEY = os.environ.get('TAVILY_API_KEY')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

# Initialize the LLM for the crew
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="moonshotai/kimi-k2-instruct"
)
````

### Original block 8

````text
multimodal_agentic_application/
├── agents.py         # Agent definitions and configurations
├── agentic_tools.py  # Tool implementations for routing and execution
├── config.py         # Configuration settings and LLM setup
├── helper_tools.py   # Core tool functions (text2image, text2speech, etc.)
├── main.py           # Main application entry point
├── tasks.py          # Task definitions for the agent workflow
├── requirements.txt  # Python dependencies
└── .env              # Secret API keys
````

### Original source artwork

![Original source figure 1](./original-assets/7773d3f4513d-01.png)

![Original source figure 2](./original-assets/7773d3f4513d-02.jpg)

![Original source figure 3](./original-assets/7773d3f4513d-03.png)